# Online Retail Dataset Analysis

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Data, First 5 Rows, Missing Values & Remove Duplicates

In [2]:
df = pd.read_excel('Online Retail.xlsx', engine='openpyxl')
print(f'Shape: {df.shape}')

print('\nFirst 5 Rows:')
print(df.head())

print('\nMissing Values:')
print(df.isnull().sum())

before = len(df)
df.drop_duplicates(inplace=True)
df.dropna(subset=['CustomerID'], inplace=True)
df['CustomerID'] = df['CustomerID'].astype(int)
print(f'\nDuplicates removed: {before - len(df)}')
print(f'Rows after cleaning: {len(df)}')

Shape: (541909, 8)

First 5 Rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55    17850.00  United Kingdom  
1 2010-12-01 08:26:00       3.39    17850.00  United Kingdom  
2 2010-12-01 08:26:00       2.75    17850.00  United Kingdom  
3 2010-12-01 08:26:00       3.39    17850.00  United Kingdom  
4 2010-12-01 08:26:00       3.39    17850.00  United Kingdom  

Missing Values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
Custom


Duplicates removed: 140305
Rows after cleaning: 401604


## 2. Generate Revenue Column (Quantity x UnitPrice)

In [3]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']
print('Revenue column created.')
print(df[['InvoiceNo', 'Description', 'Quantity', 'UnitPrice', 'Revenue']].head(10))

Revenue column created.
  InvoiceNo                          Description  Quantity  UnitPrice  Revenue
0    536365   WHITE HANGING HEART T-LIGHT HOLDER         6       2.55    15.30
1    536365                  WHITE METAL LANTERN         6       3.39    20.34
2    536365       CREAM CUPID HEARTS COAT HANGER         8       2.75    22.00
3    536365  KNITTED UNION FLAG HOT WATER BOTTLE         6       3.39    20.34
4    536365       RED WOOLLY HOTTIE WHITE HEART.         6       3.39    20.34
5    536365         SET 7 BABUSHKA NESTING BOXES         2       7.65    15.30
6    536365    GLASS STAR FROSTED T-LIGHT HOLDER         6       4.25    25.50
7    536366               HAND WARMER UNION JACK         6       1.85    11.10
8    536366            HAND WARMER RED POLKA DOT         6       1.85    11.10
9    536367        ASSORTED COLOUR BIRD ORNAMENT        32       1.69    54.08


## 3. Top 10 Selling Products Based on Quantity

In [4]:
top10 = (
    df.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top10.columns = ['Product', 'Total Quantity Sold']
top10.index = top10.index + 1
print('Top 10 Selling Products:')
print(top10.to_string())

Top 10 Selling Products:
                               Product  Total Quantity Sold
1    WORLD WAR 2 GLIDERS ASSTD DESIGNS                53119
2              JUMBO BAG RED RETROSPOT                44963
3        ASSORTED COLOUR BIRD ORNAMENT                35215
4   WHITE HANGING HEART T-LIGHT HOLDER                34128
5      PACK OF 72 RETROSPOT CAKE CASES                33386
6                       POPCORN HOLDER                30492
7                   RABBIT NIGHT LIGHT                27045
8              MINI PAINT SET VINTAGE                 25880
9           PACK OF 12 LONDON TISSUES                 25305
10  PACK OF 60 PINK PAISLEY CAKE CASES                24129


## 4. Total Revenue of the Store

In [5]:
total_revenue = df['Revenue'].sum()
print(f'Total Revenue: £{total_revenue:,.2f}')

Total Revenue: £8,278,519.42


## 5. Total Unique Customers, Count & List

In [6]:
unique_count = df['CustomerID'].nunique()
print(f'Total Unique Customers: {unique_count}')

customer_list = (
    df.groupby('CustomerID')
    .agg(
        Total_Orders=('InvoiceNo', 'nunique'),
        Total_Items=('Quantity', 'sum'),
        Total_Spent=('Revenue', 'sum')
    )
    .sort_values('Total_Spent', ascending=False)
    .reset_index()
)
customer_list.index = customer_list.index + 1
print('\nCustomer List (Top 15 by Spending):')
print(customer_list.head(15).to_string())

Total Unique Customers: 4372

Customer List (Top 15 by Spending):
    CustomerID  Total_Orders  Total_Items  Total_Spent
1        14646            77       196719    279489.02
2        18102            62        64122    256438.49
3        17450            55        69009    187322.17
4        14911           248        77155    132458.73
5        12415            26        77242    123725.45
6        14156            66        56908    113214.59
7        17511            46        63012     88125.38
8        16684            31        49390     65892.08
9        13694            60        61899     62690.54
10       15311           118        37673     59284.19
11       13089           118        30742     57322.13
12       14096            34        16335     57120.91
13       15061            55        28590     54228.74
14       16029            76        33632     53168.69
15       17949            52        27571     52750.84


## 6. Average Order Value

In [7]:
order_revenue = df.groupby('InvoiceNo')['Revenue'].sum()
aov = order_revenue.mean()

print(f'Average Order Value : £{aov:,.2f}')
print(f'Min Order Value     : £{order_revenue.min():,.2f}')
print(f'Max Order Value     : £{order_revenue.max():,.2f}')
print(f'Median Order Value  : £{order_revenue.median():,.2f}')

Average Order Value : £373.07
Min Order Value     : £-168,469.60
Max Order Value     : £168,469.60
Median Order Value  : £239.41


## 7. Country with Highest Number of Orders

In [8]:
country_orders = (
    df.groupby('Country')['InvoiceNo']
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
)
country_orders.columns = ['Country', 'Number of Orders']
country_orders.index = country_orders.index + 1

print(f'Country with Highest Orders: {country_orders.iloc[0]["Country"]}  ({country_orders.iloc[0]["Number of Orders"]:,} orders)')
print('\nAll Countries:')
print(country_orders.to_string())

Country with Highest Orders: United Kingdom  (19,857 orders)

All Countries:
                 Country  Number of Orders
1         United Kingdom             19857
2                Germany               603
3                 France               458
4                   EIRE               319
5                Belgium               119
6                  Spain               105
7            Netherlands               101
8            Switzerland                71
9               Portugal                70
10             Australia                69
11                 Italy                55
12               Finland                48
13                Sweden                46
14                Norway                40
15       Channel Islands                33
16                 Japan                28
17                Poland                24
18               Denmark                21
19                Cyprus                20
20               Austria                19
21             Singa

## 8. Variate Analysis – Products Bought Together

In [9]:
from itertools import combinations
from collections import Counter

basket = (
    df[df['Quantity'] > 0]
    .groupby('InvoiceNo')['Description']
    .apply(lambda items: list(set(items)))
)

pair_counts = Counter()
for item_list in basket:
    if len(item_list) >= 2:
        for pair in combinations(sorted(item_list), 2):
            pair_counts[pair] += 1

top_pairs = pd.DataFrame(
    [(a, b, cnt) for (a, b), cnt in pair_counts.most_common(20)],
    columns=['Product A', 'Product B', 'Times Bought Together']
)
top_pairs.index = top_pairs.index + 1
print(f'Total unique product pairs: {len(pair_counts):,}')
print('\nTop 20 Products Bought Together:')
print(top_pairs.to_string())

Total unique product pairs: 2,276,995

Top 20 Products Bought Together:
                             Product A                           Product B  Times Bought Together
1              JUMBO BAG PINK POLKADOT             JUMBO BAG RED RETROSPOT                    546
2      GREEN REGENCY TEACUP AND SAUCER    ROSES REGENCY TEACUP AND SAUCER                     541
3           ALARM CLOCK BAKELIKE GREEN           ALARM CLOCK BAKELIKE RED                     530
4              LUNCH BAG PINK POLKADOT             LUNCH BAG RED RETROSPOT                    523
5              LUNCH BAG  BLACK SKULL.             LUNCH BAG RED RETROSPOT                    517
6          WOODEN FRAME ANTIQUE WHITE    WOODEN PICTURE FRAME WHITE FINISH                    468
7              LUNCH BAG RED RETROSPOT          LUNCH BAG SPACEBOY DESIGN                     467
8              LUNCH BAG  BLACK SKULL.             LUNCH BAG PINK POLKADOT                    464
9   GARDENERS KNEELING PAD CUP OF TEA    GARDE

## 9. Customers Who Purchased More Than 10 Items

In [10]:
customer_items = (
    df.groupby('CustomerID')['Quantity']
    .sum()
    .reset_index()
)
customer_items.columns = ['CustomerID', 'Total Items Purchased']

high_volume = (
    customer_items[customer_items['Total Items Purchased'] > 10]
    .sort_values('Total Items Purchased', ascending=False)
    .reset_index(drop=True)
)
high_volume.index = high_volume.index + 1

print(f'Customers who purchased more than 10 items: {len(high_volume):,}')
print('\nTop 20:')
print(high_volume.head(20).to_string())

Customers who purchased more than 10 items: 4,286

Top 20:
    CustomerID  Total Items Purchased
1        14646                 196719
2        12415                  77242
3        14911                  77155
4        17450                  69009
5        18102                  64122
6        17511                  63012
7        13694                  61899
8        14298                  58021
9        14156                  56908
10       16684                  49390
11       15311                  37673
12       16029                  33632
13       16422                  32592
14       17404                  32324
15       16333                  32184
16       13089                  30742
17       15061                  28590
18       15769                  27660
19       17949                  27571
20       17381                  25646
